In [3]:
from sqlalchemy import create_engine
import pandas as pd
import yaml

with open("C:/Users/massi/OneDrive/Desktop/Uni/DMFBI-R/5. code/creds.yaml", "r") \
      as file:
    creds = yaml.safe_load(file)


def reads_from_mysql(creds, query):
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    df = pd.read_sql(query, engine)
    return df

def write_to_database(creds, df, table_name, if_exists='append'):
    """
    Returns as dataframe the result of a query to a MySQL database.

    Args:
        creds (dict): The credentials to access the database.
        query (string): The query.

    Returns:
        pandas dataframe: the output table of the query.
    """
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    with engine.connect() as connection:
        df.to_sql(table_name, con=connection, if_exists=if_exists, index=False) 


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('C:/Users/massi/OneDrive/Desktop/Uni/DMFBI-R/4.nocode/data/invoices_eae.csv',sep=';')

In [6]:
write_to_database(creds=creds['mysql-db'], df=df, table_name='python_test')

ASSIGNMENT

In [8]:
import pandas as pd

In [9]:
meteo_types = {'temperature':'float64','relative_humidity':'float64','precipitation_rate':'float64','wind_speed':'float64','zipcode':'str'}
contracts_types = {'CONTRACT_ID':'int64','CLIENT_TYPE_ID':'int64','AVG_EUROS_IMPORT':'float64','POWER_P1':'float64','HAS_GAS':'boolean','HAS_SOLAR':'boolean','ZIPCODE':'str'}
zipcode_types = {'ZIPCODE':'str','ZC_LATITUDE':'float64','ZC_LONGITUDE':'float64','AUTONOMOUS_COMMUNITY':'str','AUTONOMOUS_COMMUNITY_NK':'str','PROVINCE':'str'}

In [10]:
df_meteo = pd.read_csv('meteo_eae.csv', delimiter=';',dtype=meteo_types,parse_dates=['date'])
df_contracts = pd.read_csv('contracts_eae.csv', dtype=contracts_types)
df_zipcode = pd.read_csv('zipcode_eae_v2.csv', dtype=zipcode_types)

In [11]:
df_meteo.columns = df_meteo.columns.str.lower()
df_contracts.columns = df_contracts.columns.str.lower()
df_zipcode.columns = df_zipcode.columns.str.lower()

In [12]:
zipcode_top =  list(df_contracts.groupby('zipcode')['contract_id'].count().reset_index().nlargest(10,'contract_id')['zipcode'])

In [13]:
df_meteo_top = df_meteo[df_meteo['zipcode'].isin(zipcode_top)]

In [14]:
df_contracts['p1_category'] = df_contracts['power_p1'].apply(lambda x: 'Over 5 MW' if x >= 5000 else 'Under 3 MW' if x < 3000 else 'Between 3 and 5 MW')

In [15]:
df_contracts_zero = df_contracts[df_contracts['client_type_id']==0]

In [16]:
df_contracts_temperatures = df_contracts_zero.merge(df_meteo_top, how='right', left_on='zipcode', right_on='zipcode')